# Paired ontology-grounding evaluation

Run `RDFSOLVE_NOTEBOOK=04_ontology.ipynb sbatch scripts/slurm_qwen_mcp.sh` after the cache notebook. Three repeats are the default; set `RDFSOLVE_REPEATS=1` for a pilot. `RDFSOLVE_CASE` can select one case ID.

Compare the same fixed questions and RDF snapshots with grounding off/on. References stay in this notebook. Complete and failed runs both contribute to F1. These are development cases; results do not establish held-out accuracy.

Budget per run: 30 model requests, 32,768 total output tokens; each completion is capped at 8,192 tokens. Exhaustion scores zero. Temperature 0.2; paired seeds 101 onward.


In [ ]:
import hashlib, json, os, shutil
from pathlib import Path
import numpy as np
import pandas as pd
from rdfsolve.api import Client, ask_rdf
from pydantic_ai.usage import UsageLimits

root = Path(os.environ["RDFSOLVE_ROOT"])
folder, output = root / "notebooks/mcp/schemas", Path(os.environ["RDFSOLVE_OUTPUT"])
cache = root / "notebooks/mcp/ontology-cache.json"
cases = json.loads((root / "notebooks/mcp/ontology-cases.json").read_text())
if os.getenv("RDFSOLVE_CASE", "all") != "all":
    cases = [c for c in cases if c["id"] == os.environ["RDFSOLVE_CASE"]]
assert cases and cache.is_file()
repeats = int(os.getenv("RDFSOLVE_REPEATS", "3"))
base_seed = int(os.getenv("RDFSOLVE_SEED", "101"))
(output / "experiment-config.json").write_text(json.dumps({"repeats":repeats, "base_seed":base_seed, "temperature":0.2, "max_tokens":8192, "output_tokens_limit":32768, "request_limit":30}, indent=2))
inputs = [cache, root / "notebooks/mcp/ontology-cases.json"] + [folder / (c["source"] + suffix) for c in cases for suffix in (".ttl", ".schema.json")]
(output / "experiment-inputs.json").write_text(json.dumps({str(p.relative_to(root)): hashlib.sha256(p.read_bytes()).hexdigest() for p in inputs}, indent=2))

for path in inputs:
    target = output / "inputs" / path.relative_to(root)
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(path, target)
folder = output / "inputs/notebooks/mcp/schemas"
cache = output / "inputs/notebooks/mcp/ontology-cache.json"

def tuples(rows, columns):
    return {tuple(json.dumps(row.get(c), sort_keys=True, ensure_ascii=False) for c in columns) for row in rows}


In [ ]:
references = {}
for case in cases:
    source = case["source"]
    with Client.open(folder / f"{source}.schema.json", data_file=folder / f"{source}.ttl") as client:
        result = client.select(case["reference"])
        for name, query in case.get("counterexamples", {}).items():
            altered = client.select(query)
            exact = lambda answer: {tuple((key, term.type, term.value, term.lang, term.datatype) for key, term in sorted(row.items())) for row in answer.rows}
            assert exact(altered) != exact(result), f"{case['id']}: the snapshot cannot detect {name}"
    rows = [{k: {"type":v.type, "value":v.value, **({"xml:lang":v.lang} if v.lang else {}), **({"datatype":v.datatype} if v.datatype else {})} for k,v in row.items()} for row in result.rows]
    assert not any(v["type"] == "uri" and v["value"].startswith("file:") for row in rows for v in row.values()), "A reference projects a parser-derived local identity"
    assert rows, f"The development case {case['id']} needs a nonempty reference"
    references[case["id"]] = tuples(rows, case["columns"])
    (output / (case["id"] + ".reference.json")).write_text(json.dumps({"query":case["reference"], "rows":rows}, indent=2))
print({key: len(rows) for key,rows in references.items()})


In [ ]:
metrics = []
for repeat in range(repeats):
    for case in cases:
        source = case["source"]
        for enabled in ([False, True] if (base_seed+repeat) % 2 == 1 else [True, False]):
            answer = await ask_rdf(case["question"], schema=folder / f"{source}.schema.json", data_file=folder / f"{source}.ttl", graph_uris=[], source_id=source,
                ontology_grounding=enabled, ontology_cache=cache, ontology_offline=True,
                model_settings={"temperature":0.2, "seed":base_seed+repeat, "max_tokens":8192},
                usage_limits=UsageLimits(request_limit=30, output_tokens_limit=32768), output_dir=output)
            expected, actual = references[case["id"]], tuples(answer.bindings, case["columns"])
            overlap = len(expected & actual)
            precision, recall = overlap / len(actual) if actual else 0, overlap / len(expected)
            f1 = 2*precision*recall/(precision+recall) if precision+recall and answer.state == "complete" else 0
            row = dict(case=case["id"], repeat=repeat, seed=base_seed+repeat, ontology=enabled, state=answer.state,
                       expected=len(expected), actual=len(actual), precision=precision, recall=recall, f1=f1,
                       exact=answer.state=="complete" and actual==expected, diagnostics=answer.diagnostics())
            metrics.append(row)
            (output / "ontology-metrics.json").write_text(json.dumps(metrics, indent=2))
            print(case["id"], enabled, answer.state, "F1", round(f1,3), answer.text, flush=True)


In [ ]:
table = pd.DataFrame([{k:v for k,v in m.items() if k != "diagnostics"} for m in metrics])
display(table.groupby(["case","ontology"])[["f1","exact"]].mean())
paired = table.pivot(index=["case","repeat"], columns="ontology", values="f1")
delta = (paired[True] - paired[False]).groupby("case").mean()
rng = np.random.default_rng(42)
bootstrap = rng.choice(delta.to_numpy(), size=(5000,len(delta)), replace=True).mean(axis=1)
summary = {"macro_f1": table.groupby("ontology").f1.mean().to_dict(), "paired_delta": float(delta.mean()),
           "question_bootstrap_95pct": np.quantile(bootstrap,[.025,.975]).tolist(), "questions":len(delta), "repeats":repeats,
           "interpretation":"Development pilot; bootstrap by question; no held-out claim."}
(output / "ontology-summary.json").write_text(json.dumps(summary,indent=2))
display(summary)
